<a href="https://colab.research.google.com/github/TAUforPython/BioMedAI/blob/main/LMM_GNN_Fact_checking_dashbord.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CELL 1: ENVIRONMENT SETUP & IMPORTS

In [1]:

"""Core imports (all available in standard Colab)"""
import os
import re
import json
import time
import random
import logging
import warnings
from collections import Counter, defaultdict
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any
import hashlib
import functools

"""Data & ML"""
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

"""Deep Learning (pre-installed in Colab)"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

"""Transformers (pre-installed in Colab)"""
from transformers import RobertaTokenizer, RobertaModel
from torch.optim import AdamW

"""Visualization"""
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from IPython.display import display, HTML, clear_output, Javascript

"""Interactive widgets"""
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual

"""Graph processing"""
import networkx as nx

In [2]:
"""Setup"""
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("\nEnvironment ready!")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4

Environment ready!


# CELL 2: CONFIGURATION & API TOKENS


In [4]:


import os
from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_token_example") # Uncomment if you have a token
os.environ["PUBMED_API_KEY"] = userdata.get("PubMed_token")

"""Initialize configuration"""
CONFIG = {
    'MODEL_NAME': 'roberta-base',
    'MAX_LENGTH': 512,
    'BATCH_SIZE': 8,
    'LEARNING_RATE': 2e-5,
    'EPOCHS': 3,
    'HIDDEN_DIM': 64,
    'NUM_CLASSES': 2,
    'DEVICE': 'cuda' if torch.cuda.is_available() else 'cpu',
    'USE_PUBMED': True,
    'USE_GNN': True,
    'VISUALIZE': True
}

print(f"\nConfiguration:")
for k, v in CONFIG.items():
    if 'TOKEN' not in k and 'KEY' not in k:
        print(f"  {k}: {v}")
print("PUBMED_API_KEY" in os.environ)  # Returns True if key is set


Configuration:
  MODEL_NAME: roberta-base
  MAX_LENGTH: 512
  BATCH_SIZE: 8
  LEARNING_RATE: 2e-05
  EPOCHS: 3
  HIDDEN_DIM: 64
  NUM_CLASSES: 2
  DEVICE: cuda
  USE_PUBMED: True
  USE_GNN: True
  VISUALIZE: True
True


# CELL 3: OPTIMIZED MEDICAL KNOWLEDGE GRAPH


In [5]:


class MedicalKnowledgeGraph:
    """
    Streamlined Knowledge Graph for Medical Fact Checking
    Optimized for Colab memory constraints
    """
    def __init__(self):
        self.nodes = []
        self.edges = []
        self.node_features = []
        self.node_types = []
        self.node_id_map = {}
        self.node_labels = {}
        self.node_metadata = {} # Fixed typo: node_m etadata

    def __repr__(self):
        return f"<MedicalKnowledgeGraph: {len(self.nodes)} nodes, {len(self.edges)} edges>"

    def add_claim_node(self, claim_text: str, claim_id: str) -> int:
        """Add a claim node"""
        node_id = len(self.nodes)
        node_name = f"claim_{claim_id}"
        self.nodes.append(node_name)
        self.node_types.append('claim')
        self.node_labels[node_name] = claim_text[:60] + "..." if len(claim_text) > 60 else claim_text
        self.node_metadata[node_name] = {
            'full_text': claim_text,
            'type': 'claim',
            'id': claim_id
        }
        self.node_features.append(self._text_to_features(claim_text))
        self.node_id_map[node_name] = node_id
        return node_id

    def add_evidence_node(self, evidence_text: str, evidence_id: str) -> int:
        """Add an evidence node"""
        node_id = len(self.nodes)
        node_name = f"evidence_{evidence_id}"
        self.nodes.append(node_name)
        self.node_types.append('evidence')
        self.node_labels[node_name] = evidence_text[:60] + "..." if len(evidence_text) > 60 else evidence_text
        self.node_metadata[node_name] = {
            'full_text': evidence_text,
            'type': 'evidence',
            'id': evidence_id
        }
        self.node_features.append(self._text_to_features(evidence_text))
        self.node_id_map[node_name] = node_id
        return node_id

    def add_paper_node(self, paper_info: Dict, paper_id: str) -> int:
        """Add a paper node"""
        node_id = len(self.nodes)
        node_name = f"paper_{paper_id}"
        self.nodes.append(node_name)
        self.node_types.append('paper')
        title = paper_info.get('title', 'Unknown Paper') or 'Unknown Paper'
        self.node_labels[node_name] = title[:50] + "..." if len(title) > 50 else title # Fixed typo: tit le
        self.node_metadata[node_name] = {
            'title': title,
            'authors': paper_info.get('authors', []),
            'journal': paper_info.get('journal', ''),
            'year': paper_info.get('year', ''),
            'pmid': paper_info.get('pmid', ''),
            'type': 'paper',
            'relevance_score': paper_info.get('relevance_score', 0.5)
        }
        self.node_features.append(self._paper_to_features(paper_info))
        self.node_id_map[node_name] = node_id
        return node_id

    def add_entity_node(self, entity_name: str, entity_type: str) -> int:
        """Add a medical entity node"""
        node_key = f"entity_{entity_name}_{entity_type}"
        if node_key in self.node_id_map:
            return self.node_id_map[node_key]

        node_id = len(self.nodes)
        self.nodes.append(node_key)
        self.node_types.append('entity')
        display = f"{entity_name} ({entity_type})" if entity_name and entity_type else "Unknown Entity"
        self.node_labels[node_key] = display[:50] + "..." if len(display) > 50 else display
        self.node_metadata[node_key] = {
            'name': entity_name or "Unknown",
            'entity_type': entity_type or "Unknown",
            'type': 'entity'
        }
        self.node_features.append(self._entity_to_features(entity_name, entity_type))
        self.node_id_map[node_key] = node_id
        return node_id

    def add_edge(self, source_id: int, target_id: int, relation_type: str, weight: float = 1.0):
        """Add an edge between nodes"""
        self.edges.append((source_id, target_id, relation_type, weight))

    def _text_to_features(self, text: str) -> List[float]:
        """Convert text to simple feature vector"""
        if not text:
            return [0.0, 0.0, 0.0]
        words = str(text).lower().split()
        return [
            float(len(words)),
            float(len(set(words))),
            float(sum(len(w) for w in words) / len(words)) if words else 0.0
        ]

    def _paper_to_features(self, paper_info: Dict) -> List[float]:
        """Convert paper info to feature vector"""
        if not paper_info:
            return [0.0, 0.0, 0.0]
        return [
            1.0,
            float(paper_info.get('relevance_score', 0.5)),
            float(len(paper_info.get('authors', [])))
        ]

    def _entity_to_features(self, entity_name: str, entity_type: str) -> List[float]:
        """Convert entity to feature vector"""
        type_map = {'disease': 1, 'drug': 2, 'symptom': 3, 'treatment': 4, 'cancer': 5}
        return [
            float(type_map.get(entity_type, 0)),
            float(len(entity_name)) if entity_name else 0.0,
            1.0
        ]

    def to_networkx(self) -> nx.DiGraph:
        """Convert to NetworkX for visualization"""
        G = nx.DiGraph()

        # Add nodes with attributes
        for i, (node_name, node_type) in enumerate(zip(self.nodes, self.node_types)):
            G.add_node(
                node_name,
                type=node_type,
                label=self.node_labels.get(node_name, node_name),
                metadata=self.node_metadata.get(node_name, {})
            )

        # Add edges
        for source, target, relation, weight in self.edges:
            if source < len(self.nodes) and target < len(self.nodes):
                G.add_edge(
                    self.nodes[source],
                    self.nodes[target],
                    relation=relation,
                    weight=weight
                )

        return G

print("MedicalKnowledgeGraph class defined")

MedicalKnowledgeGraph class defined


# CELL 4: OPTIMIZED PUBMED INTEGRATION WITH CACHING

In [7]:


import requests
import xml.etree.ElementTree as ET
from urllib.parse import quote_plus

PUBMED_CACHE = {}

def _cache_key(query, max_results, api_key):
    return hashlib.md5(f"{query}{max_results}{api_key}".encode()).hexdigest()

@functools.lru_cache(maxsize=128)
def search_pubmed_cached(query: str, max_results: int = 5, api_key: Optional[str] = None) -> Dict:
    """Thread-safe, cached PubMed search with medical query enhancement"""
    cache_key = _cache_key(query, max_results, api_key)
    if cache_key in PUBMED_CACHE:
        return PUBMED_CACHE[cache_key]

    # Enhance medical query with MeSH-style terms
    medical_boost = "OR clinical trial OR meta-analysis OR systematic review"
    enhanced_query = f"({query}) {medical_boost}"

    # Call original search_pubmed logic
    result = search_pubmed(enhanced_query, max_results, api_key)
    PUBMED_CACHE[cache_key] = result
    return result

def search_pubmed(query: str, max_results: int = 5, api_key: Optional[str] = None) -> Dict:
    """
    Search PubMed for relevant medical literature
    Optimized for Colab with rate limiting
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {
        'db': 'pubmed',
        'term': query,
        'retmax': max_results,
        'retmode': 'json',
        'sort': 'relevance'
    }
    if api_key:
        params['api_key'] = api_key

    try:
        # Search
        search_resp = requests.get(base_url, params=params, timeout=10)
        search_resp.raise_for_status()
        search_data = search_resp.json()
        pmids = search_data.get('esearchresult', {}).get('idlist', [])

        if not pmids:
            return {
                'papers': [],
                'total_papers': 0,
                'confidence_boost': 0.0,
                'literature_evidence': 'No relevant literature found'
            }

        # Fetch details
        fetch_params = {
            'db': 'pubmed',
            'id': ','.join(pmids),
            'retmode': 'xml',
            'rettype': 'abstract'
        }
        if api_key:
            fetch_params['api_key'] = api_key

        time.sleep(0.2)  # Rate limiting

        fetch_resp = requests.get(fetch_url, params=fetch_params, timeout=10)
        fetch_resp.raise_for_status()

        # Parse XML
        papers = parse_pubmed_xml(fetch_resp.text)

        # Add relevance scores
        for i, paper in enumerate(papers):
            paper['relevance_score'] = max(0.7, 1.0 - i * 0.1)

        confidence_boost = min(0.3, len(papers) * 0.1)

        return {
            'papers': papers,
            'total_papers': len(papers),
            'confidence_boost': confidence_boost,
            'literature_evidence': f"Found {len(papers)} relevant studies. {'Strong' if len(papers) >= 2 else 'Moderate'} literature support."
        }

    except Exception as e:
        logger.warning(f"PubMed search failed: {e}")
        return {
            'papers': [],
            'total_papers': 0,
            'confidence_boost': 0.0,
            'literature_evidence': f'Search error: {str(e)}'
        }

def parse_pubmed_xml(xml_content: str) -> List[Dict]:
    """Parse PubMed XML response"""
    try:
        root = ET.fromstring(xml_content)
        papers = []
        for article in root.findall('.//PubmedArticle'):
            pmid_elem = article.find('.//PMID')
            pmid = pmid_elem.text if pmid_elem is not None else 'Unknown'

            title_elem = article.find('.//ArticleTitle')
            title = title_elem.text if title_elem is not None else 'Unknown Title'

            authors = []
            for author in article.findall('.//Author'):
                last = author.find('LastName')
                first = author.find('ForeName')
                if last is not None and first is not None and last.text and first.text:
                    authors.append(f"{first.text} {last.text}")

            journal_elem = article.find('.//Journal/Title')
            journal = journal_elem.text if journal_elem is not None else 'Unknown'

            year_elem = article.find('.//PubDate/Year')
            year = year_elem.text if year_elem is not None else 'Unknown'

            papers.append({
                'pmid': pmid,
                'title': title,
                'authors': authors,
                'journal': journal,
                'year': year
            })

        return papers
    except Exception as e:
        logger.warning(f"XML parsing failed: {e}")
        return []

def extract_medical_entities_v2(text: str) -> List[Tuple[str, str]]:
    """Regex-based, boundary-aware medical entity extraction"""
    if not text: return []
    text = text.lower()
    patterns = {
        "disease": r"\b(?:cancer|diabetes|hypertension|alzheimer|asthma|infection|syndrome|disorder)\b",
        "treatment": r"\b(?:vaccine|therapy|medication|drug|surgery|radiation|antibiotic|chemotherapy)\b",
        "symptom": r"\b(?:pain|fever|fatigue|nausea|cough|headache|inflammation|swelling)\b",
        "anatomy": r"\b(?:heart|lung|liver|brain|kidney|blood|cell|tissue)\b"
    }
    entities = []
    for etype, pat in patterns.items():
        for match in re.finditer(pat, text):
            entities.append((match.group(0), etype))
    return list(set(entities))

print("PubMed integration ready")

PubMed integration ready


# CELL 5: GNN MODEL (Pure PyTorch)

In [8]:
class SimpleGNN(nn.Module):
    """
    Graph Neural Network using pure PyTorch
    Simulates GCN behavior without PyTorch Geometric
    """
    def __init__(self, num_node_features: int = 3, hidden_dim: int = 64):
        super(SimpleGNN, self).__init__()

        self.hidden_dim = hidden_dim

        # Message passing layers
        self.conv1 = nn.Linear(num_node_features, hidden_dim)
        self.conv2 = nn.Linear(hidden_dim, hidden_dim)

        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

        # Output projection to hidden_dim (not num_classes)
        self.output_proj = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3)
        )

    def forward(self, node_features: torch.Tensor, adjacency_matrix: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            graph_embedding: [hidden_dim]
            attn_weights: [num_nodes, 1]
        """
        x = node_features

        # First GCN layer
        x = self.conv1(x)
        x = F.relu(x)

        # Message passing
        if adjacency_matrix is not None:
            x = torch.matmul(adjacency_matrix, x)

        x = F.dropout(x, p=0.3, training=self.training)

        # Second GCN layer
        x = self.conv2(x)
        x = F.relu(x)

        # Graph-level attention pooling
        attn_weights = F.softmax(self.attention(x), dim=0)
        graph_embedding = torch.sum(attn_weights * x, dim=0)

        # Project to consistent hidden dimension
        output = self.output_proj(graph_embedding)

        return output, attn_weights

class HybridFactChecker(nn.Module):
    """
    Hybrid model combining RoBERTa + GNN with cross-attention
    FIXED: Dimension handling for batch processing
    """
    def __init__(self, roberta_model: nn.Module, gnn_model: nn.Module,
                 roberta_hidden: int = 768, gnn_hidden: int = 64,
                 fusion_hidden: int = 64, num_classes: int = 2):
        super(HybridFactChecker, self).__init__()

        self.roberta = roberta_model
        self.gnn = gnn_model
        self.gnn_hidden = gnn_hidden
        self.fusion_hidden = fusion_hidden

        # Freeze RoBERTa for efficiency
        for param in self.roberta.parameters():
            param.requires_grad = False

        # Projection layers
        self.roberta_proj = nn.Linear(roberta_hidden, fusion_hidden) # Fixed typo: Line ar
        self.gnn_proj = nn.Linear(gnn_hidden, fusion_hidden)

        # Cross-attention
        self.cross_attn = nn.Sequential(
            nn.Linear(fusion_hidden * 2, fusion_hidden), # Fixed typo: fusion_hidd en
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(fusion_hidden, 2)
        )

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(fusion_hidden, fusion_hidden // 2), # Fixed typo: n n.Linear
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(fusion_hidden // 2, num_classes)
        )

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                node_features: torch.Tensor, adjacency: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            input_ids: [batch_size, seq_len]
            attention_mask: [batch_size, seq_len]
            node_features: [num_nodes, num_node_features] or [batch_size, num_nodes,  num_node_features]
            adjacency: [num_nodes, num_nodes] or [batch_size, num_nodes, num_nodes]
        Returns:
            logits: [batch_size, num_classes]
            attn_scores: [batch_size, 2]
        """
        batch_size = input_ids.size(0)

        # RoBERTa encoding (no gradients)
        with torch.no_grad():
            roberta_outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask) # Fixed typo: att ention_mask
            roberta_embedding = roberta_outputs.last_hidden_state[:, 0, :]  # [batch_size, 768]

        roberta_projected = self.roberta_proj(roberta_embedding)  # [batch_size, fusion_hidden]

        # GNN encoding
        if node_features.dim() == 2:
            # Single graph: [num_nodes, features]
            gnn_output, gnn_attn = self.gnn(node_features, adjacency)  # [hidden_dim]
            gnn_projected = self.gnn_proj(gnn_output)  # [fusion_hidden]
            gnn_projected = gnn_projected.unsqueeze(0).expand(batch_size, -1)  # [batch_size, fusion_hidden]
        elif node_features.dim() == 3:
            # Batched graphs
            gnn_outputs = []
            for i in range(batch_size):
                g_out, _ = self.gnn(node_features[i], adjacency[i] if adjacency is not None else None)
                gnn_outputs.append(g_out)
            gnn_output = torch.stack(gnn_outputs)  # [batch_size, hidden_dim]
            gnn_projected = self.gnn_proj(gnn_output)  # [batch_size, fusion_hidden]
        else:
            raise ValueError(f"Unexpected node_features dimensions: {node_features.dim()}")

        # Cross-attention fusion
        combined = torch.cat([roberta_projected, gnn_projected], dim=1)  # [batch_size, fusion_hidden*2]
        attn_logits = self.cross_attn(combined)  # [batch_size, 2]
        attn_scores = F.softmax(attn_logits, dim=1)  # [batch_size, 2]

        # Weighted fusion
        text_weight = attn_scores[:, 0:1]  # [batch_size, 1]
        graph_weight = attn_scores[:, 1:2]  # [batch_size, 1]

        fused = text_weight * roberta_projected + graph_weight * gnn_projected  # [batch_size, fusion_hidden]

        # Classification
        logits = self.classifier(fused)  # [batch_size, num_classes]

        return logits, attn_scores

print("GNN and Hybrid models defined")

GNN and Hybrid models defined


# CELL 6: INTERACTIVE COLAB DASHBOARD

In [9]:
class ColabDashboard:
    """
    Interactive dashboard optimized for Google Colab
    """
    def __init__(self, config):
        self.device = torch.device(config['DEVICE'])
        self.config = config
        self.tokenizer = None
        self.roberta = None
        self.gnn = None
        self.hybrid_model = None
        self.investigation_history = []

    def initialize_models(self):
        """Load models with progress tracking"""
        print("🔬 Initializing MCI-Hub Intelligence Engine...")

        # Load tokenizer and RoBERTa
        print("  📥 Loading RoBERTa...")
        self.tokenizer = RobertaTokenizer.from_pretrained(self.config['MODEL_NAME'])
        self.roberta = RobertaModel.from_pretrained(self.config['MODEL_NAME']).to(self.device)
        self.roberta.eval()

        # Initialize GNN
        print("  🧠 Initializing Graph Neural Network...")
        self.gnn = SimpleGNN(
            num_node_features=3,
            hidden_dim=self.config['HIDDEN_DIM']
        ).to(self.device)

        # Initialize Hybrid model
        print("  ⚡ Building Hybrid Fusion Model...")
        self.hybrid_model = HybridFactChecker(
            roberta_model=self.roberta,
            gnn_model=self.gnn,
            roberta_hidden=768,
            gnn_hidden=self.config['HIDDEN_DIM'],
            fusion_hidden=self.config['HIDDEN_DIM'],
            num_classes=self.config['NUM_CLASSES']
        ).to(self.device)
        self.hybrid_model.eval()

        print("✅ All models loaded successfully!")

    def create_input_panel(self):
        """Create interactive input panel"""
        # Claim input
        self.claim_input = widgets.Textarea(
            value='',
            placeholder='Enter medical claim to investigate...\nExample: "Vitamin C prevents the common cold"',
            description='Claim:',
            layout=widgets.Layout(width='100%', height='100px'),
            style={'description_width': '80px'}
        )

        # Quick templates
        templates = [
            "Vaccines cause autism",
            "Exercise reduces cardiovascular risk",
            "Coffee prevents Parkinson's disease",
            "Sugar causes hyperactivity in children",
            "Mediterranean diet promotes longevity"
        ]

        template_box = widgets.HBox([
            widgets.Button(
                description=t[:30] + "..." if len(t) > 30 else t,
                button_style='info',
                layout=widgets.Layout(width='auto', margin='2px')
            ) for t in templates
        ])

        for btn, template in zip(template_box.children, templates):
            btn.on_click(lambda b, t=template: self._set_claim(t))

        # Investigation button
        self.investigate_btn = widgets.Button(
            description='🕵️ START INVESTIGATION',
            button_style='success',
            layout=widgets.Layout(width='100%', height='50px')
        )
        self.investigate_btn.on_click(self._run_investigation)

        # Output area
        self.output_area = widgets.Output()

        # Assemble panel
        input_panel = widgets.VBox([
            widgets.HTML("<h2>📋 Open a New Case File</h2>"),
            self.claim_input,
            widgets.HTML("<h4>Quick Templates:</h4>"),
            template_box,
            widgets.HTML("<br>"),
            self.investigate_btn,
            widgets.HTML("<hr>"),
            self.output_area
        ])

        return input_panel

    def _set_claim(self, text: str):
        """Set claim from template"""
        self.claim_input.value = text

    def _run_investigation(self, b):
        """Execute investigation pipeline"""
        claim = self.claim_input.value.strip()
        if not claim:
            with self.output_area:
                clear_output()
                print("⚠️ Please enter a claim first!")
            return

        with self.output_area:
            clear_output()

            # Progress tracking
            progress = widgets.IntProgress(
                value=0, min=0, max=5,
                description='Progress:',
                bar_style='info'
            )
            status = widgets.HTML("🔍 Initializing investigation...")
            display(widgets.VBox([status, progress]))

            # Step 1: PubMed Search
            status.value = "<b>🔍 STEP 1:</b> Searching PubMed Literature..."
            papers, evidence_text = self._retrieve_real_evidence(claim) # Changed from _simulate_evidence_retrieval
            progress.value = 1
            time.sleep(0.5)

            # Step 2: Build Knowledge Graph
            status.value = "<b>🕸️ STEP 2:</b> Building Evidence Knowledge Graph..."
            kg = self._build_knowledge_graph(claim, evidence_text, papers)
            progress.value = 2
            time.sleep(0.5)

            # Step 3: Extract Graph Features
            status.value = "<b>🧠 STEP 3:</b> Extracting Graph Features..."
            node_features = self._prepare_graph_features(kg)
            progress.value = 3
            time.sleep(0.5)

            # Step 4: Model Inference
            status.value = "<b>⚡ STEP 4:</b> Running Hybrid Model Inference..."
            verdict, confidence, attention = self._run_model(claim, evidence_text, node_features)
            progress.value = 4
            time.sleep(0.5)

            # Step 5: Render Results
            status.value = "<b>✅ STEP 5:</b> Rendering Investigation Results..."
            self._render_results(claim, verdict, confidence, attention, kg, papers)
            progress.value = 5

            # Store in history
            self.investigation_history.append({
                'timestamp': datetime.now().strftime('%H:%M:%S'),
                'claim': claim,
                'verdict': verdict,
                'confidence': confidence
            })

    def _retrieve_real_evidence(self, claim: str) -> Tuple[List[Dict], str]:
        """Replaces hardcoded simulation with live PubMed integration"""
        # Use cached search
        result = search_pubmed_cached(claim, max_results=5, api_key=os.environ.get("PUBMED_API_KEY"))

        papers = result.get("papers", [])
        evidence_text = result.get("literature_evidence", "No literature found.")

        if not papers:
            # Fallback to simulated evidence if API fails
            evidence_text = "Limited specific studies found. General medical consensus may apply."

        return papers, evidence_text

    def _build_knowledge_graph(self, claim: str, evidence: str, papers: List[Dict]) -> 'MedicalKnowledgeGraph':
        """Build knowledge graph for visualization"""
        kg = MedicalKnowledgeGraph()

        # Add nodes
        claim_id = kg.add_claim_node(claim, "main")
        evidence_id = kg.add_evidence_node(evidence, "main")
        kg.add_edge(claim_id, evidence_id, 'supported_by', 1.0)

        # Add papers
        for i, paper in enumerate(papers):
            paper_id = kg.add_paper_node(paper, f"p{i}")
            kg.add_edge(evidence_id, paper_id, 'cites', paper['relevance_score'])
            kg.add_edge(paper_id, claim_id, 'supports', paper['relevance_score'] * 0.8)

        # Add entities
        entities = extract_medical_entities_v2(claim + " " + evidence)
        for entity_name, entity_type in entities:
            entity_id = kg.add_entity_node(entity_name, entity_type)
            kg.add_edge(claim_id, entity_id, 'mentions', 0.7)
            kg.add_edge(entity_id, evidence_id, 'appears_in', 0.6)

        return kg

    def _prepare_graph_features(self, kg) -> torch.Tensor:
        """Prepare graph features for model input"""
        if kg.node_features:
            features = torch.tensor(kg.node_features, dtype=torch.float32)
            # Pad to consistent size (10 nodes max)
            if len(features) < 10:
                padding = torch.zeros(10 - len(features), 3)
                features = torch.cat([features, padding], dim=0)
            else:
                features = features[:10]
            return features.to(self.device)
        return torch.zeros(10, 3).to(self.device)

    def _run_model(self, claim: str, evidence: str, node_features: torch.Tensor) -> Tuple[str, float, float]:
        """Run hybrid model inference"""
        # Prepare text input
        text = f"Claim: {claim} Evidence: {evidence}"
        encoding = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding='max_length',
            max_length=self.config['MAX_LENGTH']
        )

        input_ids = encoding['input_ids'].to(self.device)
        attention_mask = encoding['attention_mask'].to(self.device)

        # Create simple adjacency (fully connected for demo)
        num_nodes = node_features.size(0)
        adjacency = torch.ones(num_nodes, num_nodes).to(self.device) / num_nodes

        with torch.no_grad():
            logits, attn_scores = self.hybrid_model(
                input_ids, attention_mask, node_features, adjacency
            )

            probs = F.softmax(logits, dim=-1)
            confidence = torch.max(probs).item()
            prediction = torch.argmax(logits, dim=-1).item()

            # Get attention weight for text vs graph
            text_attention = attn_scores[0, 0].item()

            verdict = "SUPPORTED" if prediction == 1 else "REFUTED"

        return verdict, confidence, text_attention

    def _render_results(self, claim: str, verdict: str, confidence: float,
                       attention: float, kg, papers: List[Dict]):
        """Render investigation results with visualizations"""

        clear_output()

        # 1. VERDICT HEADER
        verdict_color = "#48bb78" if verdict == "SUPPORTED" else "#f56565"
        verdict_icon = "✅" if verdict == "SUPPORTED" else "❌"

        display(HTML(f"""
        <div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 100%);
                   padding: 20px; border-radius: 15px; margin: 10px 0;
                   border: 2px solid {verdict_color}; box-shadow: 0 0 20px {verdict_color}40;">
            <h1 style="color: {verdict_color}; text-align: center; margin: 0;">
                {verdict_icon} {verdict}
            </h1>
            <h2 style="color: white; text-align: center; margin: 10px 0;">
                Confidence: {confidence*100:.1f}%
            </h2>
            <p style="color: #a0aec0; text-align: center;">
                Text Attention: {attention:.2f} | Graph Attention: {1-attention:.2f}
            </p>
        </div>
        """))

        # 2. OPTIMIZED KNOWLEDGE GRAPH VISUALIZATION
        print("\n🕸️ KNOWLEDGE GRAPH EXPLORER")
        print("-" * 50)
        self.render_optimized_graph(kg)

        # 3. EVIDENCE TABLE
        print("\n📚 EVIDENCE SOURCES")
        print("-" * 50)

        if papers:
            evidence_df = pd.DataFrame([
                {
                    'Title': p.get('title', 'Unknown')[:50] + "...",
                    'Year': p.get('year', 'N/A'),
                    'Relevance': f"{p.get('relevance_score', 0.5):.2f}",
                    'Authors': ", ".join(p.get('authors', [])[:2])
                } for p in papers
            ])
            display(evidence_df.style.background_gradient(subset=['Relevance'], cmap='YlGn'))
        else:
            print("No specific papers found for this claim.")

        # 4. INVESTIGATION METRICS
        print("\n📊 INVESTIGATION METRICS")
        print("-" * 50)

        metrics = {
            'Claim Complexity': len(claim.split()),
            'Evidence Sources': len(papers),
            'Graph Nodes': len(kg.nodes),
            'Graph Edges': len(kg.edges),
            'Entities Found': len([n for n in kg.node_types if n == 'entity'])
        }

        for metric, value in metrics.items():
            print(f"  • {metric}: {value}")

        # 5. ATTENTION VISUALIZATION
        print("\n🧠 ATTENTION ANALYSIS")
        print("-" * 50)

        attn_fig = go.Figure(data=[
            go.Bar(
                x=['Text (RoBERTa)', 'Graph (GNN)'],
                y=[attention, 1-attention],
                marker_color=['#63b3ed', '#48bb78'],
                text=[f"{attention:.2f}", f"{1-attention:.2f}"],
                textposition='auto'
            )
        ])

        attn_fig.update_layout(
            title='Cross-Modal Attention Weights',
            yaxis=dict(title='Weight', range=[0, 1]),
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0.1)',
            font=dict(color='white'),
            height=300
        )

        attn_fig.show()

    def render_optimized_graph(self, kg):
        """Enhanced medical knowledge graph visualization for Colab"""
        G = kg.to_networkx()
        if G.number_of_nodes() == 0:
            print("⚠️ Empty knowledge graph.")
            return

        # Better force-directed layout
        pos = nx.spring_layout(G, k=0.8, iterations=120, seed=42, scale=1.5)

        # Dynamic node sizing by degree centrality
        centrality = nx.degree_centrality(G)
        min_s, max_s = 12, 38
        max_c = max(centrality.values()) if centrality else 1
        # Create a mapping from node name to its index for node_sizes lookup
        node_to_index = {node: i for i, node in enumerate(G.nodes())}
        node_sizes = [min_s + (centrality[n] / max_c) * (max_s - min_s) for n in G.nodes()]

        fig = go.Figure()

        # 1. Edges with relation types & weighted thickness
        for u, v, data in G.edges(data=True):
            x0, y0 = pos[u]
            x1, y1 = pos[v]
            relation = data.get("relation", "linked")
            weight = data.get("weight", 1.0)
            fig.add_trace(go.Scatter(
                x=[x0, x1, None], y=[y0, y1, None],
                mode="lines",
                line=dict(color="rgba(180,180,180,0.5)", width=weight * 2.5),
                hovertext=f"<b>{relation}</b><br>Strength: {weight:.2f}",
                hoverinfo="text",
                showlegend=False
            ))

        # 2. Nodes by type with metadata-rich hover
        type_colors = {"claim": "#FF6B6B", "evidence": "#4ECDC4", "paper": "#45B7D1", "entity": "#96CEB4"}
        node_types = nx.get_node_attributes(G, "type")

        for n_type, color in type_colors.items():
            nodes = [n for n in G.nodes() if node_types.get(n) == n_type]
            if not nodes: continue

            x = [pos[n][0] for n in nodes]
            y = [pos[n][1] for n in nodes]
            labels = [G.nodes[n].get("label", n) for n in nodes]

            hover_texts = []
            for n in nodes:
                meta = G.nodes[n].get("metadata", {})
                h = f"<b>{n_type.upper()}</b>: {meta.get('full_text', meta.get('title', n))}"
                if n_type == "paper":
                    h += f"<br>📅 {meta.get('year', 'N/A')} | 🔍 Relevance: {meta.get('relevance_score', 0):.2f}"
                hover_texts.append(h)

            # Corrected line: Use the node_to_index map to get the correct size
            sizes_for_nodes = [node_sizes[node_to_index[n]] for n in nodes]

            fig.add_trace(go.Scatter(
                x=x, y=y, mode="markers+text",
                marker=dict(size=sizes_for_nodes, color=color,
                           line=dict(width=2, color="white")),
                text=labels, textposition="top center",
                textfont=dict(size=9, color="white"),
                hovertext=hover_texts, hoverinfo="text",
                name=n_type.capitalize()
            ))

        fig.update_layout(
            title=dict(text="🕸️ Medical Evidence Knowledge Graph", font=dict(color="white", size=16)),
            showlegend=True, legend=dict(font=dict(color="white"), yanchor="top", y=0.99, xanchor="left", x=0.01),
            paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(20,20,30,0.5)",
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-2, 2]),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False, range=[-2, 2]),
            height=550, margin=dict(l=10, r=10, t=40, b=10),
            dragmode="pan"  # Smoother interaction
        )
        fig.show()


# CELL 7: INITIALIZE AND RUN DASHBOARD

In [10]:
dashboard = ColabDashboard(CONFIG)
dashboard.initialize_models()
print("\n" + "="*60)
print("MEDICAL CLAIMS INTELLIGENCE HUB - READY")
print("="*60)
# Display input panel
display(dashboard.create_input_panel())

🔬 Initializing MCI-Hub Intelligence Engine...
  📥 Loading RoBERTa...


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  🧠 Initializing Graph Neural Network...
  ⚡ Building Hybrid Fusion Model...
✅ All models loaded successfully!

MEDICAL CLAIMS INTELLIGENCE HUB - READY


# CELL 8: ADVANCED VISUALIZATIONS & EXPORT

In [11]:
def create_investigation_report(history: List[Dict]):
    """
    Generate comprehensive investigation report
    """
    if not history:
        print("No investigations yet!")
        return
    df = pd.DataFrame(history)

    # Summary statistics
    total = len(df)
    supported = len(df[df['verdict'] == 'SUPPORTED'])
    refuted = len(df[df['verdict'] == 'REFUTED'])
    avg_confidence = df['confidence'].mean()

    print("="*60)
    print("📊 INVESTIGATION REPORT")
    print("="*60)
    print(f"Total Investigations: {total}")
    print(f"Supported Claims: {supported} ({supported/total*100:.1f} %)")
    print(f"Refuted Claims: {refuted} ({refuted/total*100:.1f} %)")
    print(f"Average Confidence: {avg_confidence*100:.1f}%")
    print("="*60)

    # Common layout for dark theme
    common_layout = dict(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0.1)',
        font=dict(color='white'),
        title_font_size=18
    )

    # 1. Investigations Over Time (Confidence Timeline)
    fig_timeline = px.line(
        df, y='confidence', x=df.index, title='Investigations Over Time (Confidence)',
        markers=True, line_shape='linear', color_discrete_sequence=['#63b3ed']
    )
    fig_timeline.update_traces(
        marker=dict(size=10, color=df['verdict'].map({'SUPPORTED': '#48bb78', 'REFUTED': '#f56565'})),
        line=dict(width=3)
    )
    fig_timeline.update_layout(common_layout, title_text='📈 Investigations Over Time')
    display(fig_timeline)

    # 2. Verdict Distribution (Pie Chart)
    verdict_counts = df['verdict'].value_counts().reset_index()
    verdict_counts.columns = ['Verdict', 'Count']
    fig_pie = px.pie(
        verdict_counts, values='Count', names='Verdict', title='Verdict Distribution',
        color_discrete_sequence=['#48bb78', '#f56565']
    )
    fig_pie.update_layout(common_layout, title_text='⚖️ Verdict Distribution')
    display(fig_pie)

    # 3. Confidence Trend (Rolling Average)
    df['rolling_avg_confidence'] = df['confidence'].rolling(window=3, min_periods=1).mean()
    fig_trend = px.line(
        df, y='rolling_avg_confidence', x=df.index, title='Confidence Trend (Rolling Average)',
        line_shape='spline', color_discrete_sequence=['#FFEAA7']
    )
    fig_trend.update_layout(common_layout, title_text='📉 Confidence Trend')
    display(fig_trend)

    # 4. Investigation Details (Table)
    fig_table = go.Figure(data=[go.Table(
        header=dict(
            values=['Time', 'Claim', 'Verdict', 'Confidence'],
            fill_color='rgba(0,0,0,0.2)',
            font=dict(color='white', size=12),
            align='left'
        ),
        cells=dict(
            values=[
                df['timestamp'],
                df['claim'].str[:60] + "...", # Increased claim preview length
                df['verdict'],
                (df['confidence']*100).round(1).astype(str) + "%"
            ],
            fill_color='rgba(0,0,0,0.1)',
            font=dict(color='white', size=10),
            align='left',
            height=30
        )
    )])
    fig_table.update_layout(
        title_text="📋 Investigation Details",
        title_font_size=18,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0.1)',
        font=dict(color='white')
    )
    display(fig_table)

# Add to dashboard
def show_history_button(dashboard):
    """Add history button to dashboard"""
    history_btn = widgets.Button(
        description='📊 View Investigation History',
        button_style='info'
    )
    def on_click(b):
        with dashboard.output_area:
            clear_output(wait=True)
            create_investigation_report(dashboard.investigation_history)

    history_btn.on_click(on_click)
    return history_btn

# Display history button
display(show_history_button(dashboard))

Button(button_style='info', description='📊 View Investigation History', style=ButtonStyle())

# CELL 9: EXPORT RESULTS

In [19]:
!pip install fpdf2 Pillow kaleido

import io
from PIL import Image
import plotly.io as pio
from fpdf import FPDF
from IPython.display import display, HTML, clear_output, Javascript
import ipywidgets as widgets
from typing import List, Dict
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from google.colab import files
import os


def _get_report_figures_and_table_data(history: List[Dict]):
    """
    Generates Plotly figures and the DataFrame for the report table without displaying them.
    Returns (fig_timeline, fig_pie, fig_trend, df_table_for_report).
    """
    if not history:
        return None, None, None, None

    df = pd.DataFrame(history)

    common_layout = dict(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0.1)',
        font=dict(color='white'),
        title_font_size=18
    )

    # 1. Investigations Over Time (Confidence Timeline)
    fig_timeline = px.line(
        df, y='confidence', x=df.index, title='Investigations Over Time (Confidence)',
        markers=True, line_shape='linear', color_discrete_sequence=['#63b3ed']
    )
    fig_timeline.update_traces(
        marker=dict(size=10, color=df['verdict'].map({'SUPPORTED': '#48bb78', 'REFUTED': '#f56565'})),
        line=dict(width=3)
    )
    fig_timeline.update_layout(common_layout, title_text='📈 Investigations Over Time')

    # 2. Verdict Distribution (Pie Chart)
    verdict_counts = df['verdict'].value_counts().reset_index()
    verdict_counts.columns = ['Verdict', 'Count']
    fig_pie = px.pie(
        verdict_counts, values='Count', names='Verdict', title='Verdict Distribution',
        color_discrete_sequence=['#48bb78', '#f56565']
    )
    fig_pie.update_layout(common_layout, title_text='⚖️ Verdict Distribution')

    # 3. Confidence Trend (Rolling Average)
    df['rolling_avg_confidence'] = df['confidence'].rolling(window=3, min_periods=1).mean()
    fig_trend = px.line(
        df, y='rolling_avg_confidence', x=df.index, title='Confidence Trend (Rolling Average)',
        line_shape='spline', color_discrete_sequence=['#FFEAA7']
    )
    fig_trend.update_layout(common_layout, title_text='📉 Confidence Trend')

    # 4. Investigation Details (Table Data)
    # Prepare a simplified DataFrame for the PDF table
    df_table_for_report = df[['timestamp', 'claim', 'verdict', 'confidence']].copy()
    df_table_for_report['claim'] = df_table_for_report['claim'].str[:70] + '...' # Truncate for PDF
    df_table_for_report['confidence'] = (df_table_for_report['confidence'] * 100).round(1).astype(str) + '%'

    return fig_timeline, fig_pie, fig_trend, df_table_for_report


def export_investigation_results(history: List[Dict], format: str = 'csv'):
    """
    Export investigation results to file
    """
    if not history:
        print("No results to export!")
        return
    df = pd.DataFrame(history)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    if format == 'csv':
        filename = f"mci_hub_results_{timestamp}.csv"
        df.to_csv(filename, index=False)
        print(f"✅ Results saved to {filename}")
        files.download(filename)

    elif format == 'json':
        filename = f"mci_hub_results_{timestamp}.json"
        df.to_json(filename, orient='records', indent=2)
        print(f"✅ Results saved to {filename}")
        files.download(filename)

    elif format == 'html':
        filename = f"mci_hub_results_{timestamp}.html"
        html_content = f"""
        <html>
        <head><title>MCI-Hub Investigation Results</title></head>
        <body style="font-family: Arial, sans-serif; max-width: 800px; margin: 0 auto;">
            <h1>🔬 Medical Claims Intelligence Hub - Results</h1>
            <p>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
            <hr>
           {df.to_html(index=False)}
        </body>
        </html>
        """
        with open(filename, 'w') as f:
            f.write(html_content)
        print(f"✅ Results saved to {filename}")
        files.download(filename)


def export_full_report_pdf(history: List[Dict]):
    """
    Exports the full investigation report (including plots and table) as a PDF.
    """
    if not history:
        print("No investigations yet to export to PDF!")
        return

    fig_timeline, fig_pie, fig_trend, df_table_for_report = _get_report_figures_and_table_data(history)

    if fig_timeline is None: # Means history was empty
        print("No report data to generate PDF.")
        return

    pdf = FPDF(orientation='P', unit='mm', format='A4') # Portrait, millimeters, A4 size
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()

    # Set font for titles and main text
    pdf.set_font('Arial', 'B', 16)
    pdf.cell(0, 10, 'Medical Claims Intelligence Hub - Full Report', 0, 1, 'C')
    pdf.ln(5)

    pdf.set_font('Arial', '', 10)
    pdf.cell(0, 10, f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", 0, 1, 'C')
    pdf.ln(10)

    # Summary statistics
    total = len(history)
    supported = len([h for h in history if h['verdict'] == 'SUPPORTED'])
    refuted = len([h for h in history if h['verdict'] == 'REFUTED'])
    avg_confidence = pd.DataFrame(history)['confidence'].mean() if history else 0.0

    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 7, 'Report Summary:', 0, 1)
    pdf.set_font('Arial', '', 10)
    pdf.multi_cell(0, 5, f"Total Investigations: {total}\nSupported Claims: {supported} ({supported/total*100:.1f} %)\nRefuted Claims: {refuted} ({refuted/total*100:.1f} %)\nAverage Confidence: {avg_confidence*100:.1f}%")
    pdf.ln(5)

    # Add Plots
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 7, 'Visualizations:', 0, 1)
    pdf.ln(2)

    plot_figs = [
        (fig_timeline, 'Investigations Over Time'),
        (fig_pie, 'Verdict Distribution'),
        (fig_trend, 'Confidence Trend')
    ]

    for i, (fig, title) in enumerate(plot_figs):
        pdf.set_font('Arial', 'I', 10)
        pdf.cell(0, 5, f"Figure {i+1}: {title}", 0, 1)
        pdf.ln(1)

        # Convert plotly figure to PNG bytes and then add to PDF
        img_bytes = pio.to_image(fig, format='png', width=900, height=500, scale=2)
        img_data = io.BytesIO(img_bytes)
        # Save temporary image to disk for FPDF (workaround for direct BytesIO image import issues)
        temp_img_path = f"temp_plot_{i}.png"
        with open(temp_img_path, "wb") as f: f.write(img_data.getvalue())

        # Add image, width 180mm (A4 width is 210mm, with 15mm margins on each side -> 180mm usable)
        pdf.image(temp_img_path, x=15, w=180)
        pdf.ln(5)
        os.remove(temp_img_path) # Clean up temporary file

    # Add Table
    pdf.add_page()
    pdf.set_font('Arial', 'B', 12)
    pdf.cell(0, 10, 'Investigation Details Table:', 0, 1)
    pdf.ln(2)

    # Table headers
    pdf.set_font('Arial', 'B', 8)
    col_widths = [20, 110, 20, 20] # Adjust widths based on content
    headers = ['Time', 'Claim', 'Verdict', 'Confidence']
    for i, header in enumerate(headers):
        pdf.cell(col_widths[i], 7, header, 1, 0, 'C')
    pdf.ln()

    # Table rows
    pdf.set_font('Arial', '', 8)
    for index, row in df_table_for_report.iterrows():
        for i, col in enumerate(['timestamp', 'claim', 'verdict', 'confidence']):
            # Ensure text fits within cell width
            text = str(row[col])
            if col == 'claim':
                # For claim, use multi_cell if needed
                x_pos = pdf.get_x()
                y_pos = pdf.get_y()
                pdf.multi_cell(col_widths[i], 5, text, 1, 'L', 0)
                # Reset position for next cell in row, assuming fixed height
                pdf.set_xy(x_pos + col_widths[i], y_pos)
            else:
                pdf.cell(col_widths[i], 10, text, 1, 0, 'L')
        pdf.ln()

    # Output PDF
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"mci_hub_full_report_{timestamp}.pdf"
    pdf.output(filename, 'F')
    print(f"✅ Full report saved to {filename}")
    files.download(filename)


# Export buttons
export_csv = widgets.Button(description='📥 Export CSV', button_style='primary')
export_json = widgets.Button(description='📥 Export JSON', button_style='primary')
export_html = widgets.Button(description='📥 Export HTML Report', button_style='success')
export_pdf = widgets.Button(description='📥 Export PDF Report', button_style='danger') # New PDF button

export_csv.on_click(lambda b: export_investigation_results(dashboard.investigation_history, 'csv'))
export_json.on_click(lambda b: export_investigation_results(dashboard.investigation_history, 'json'))
export_html.on_click(lambda b: export_investigation_results(dashboard.investigation_history, 'html'))
export_pdf.on_click(lambda b: export_full_report_pdf(dashboard.investigation_history)) # New PDF button handler

display(widgets.HBox([export_csv, export_json, export_html, export_pdf]))


ValueError: 
Image export using the "kaleido" engine requires the kaleido package,
which can be installed using pip:
    $ pip install -U kaleido


In [18]:
!pip install kaleido -q

In [16]:
import io
from PIL import Image
import plotly.io as pio
from IPython.display import display, HTML, clear_output, Javascript
import ipywidgets as widgets # Keep this import for other widgets if any, or for consistency
from typing import List, Dict # For type hints
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.colab import output # Import for Colab-specific output callbacks

def create_investigation_report_static(history: List[Dict]):
    """
    Generate comprehensive investigation report with static images
    """
    if not history:
        print("No investigations yet!")
        return
    df = pd.DataFrame(history)

    # Summary statistics
    total = len(df)
    supported = len(df[df['verdict'] == 'SUPPORTED'])
    refuted = len(df[df['verdict'] == 'REFUTED'])
    avg_confidence = df['confidence'].mean()

    print("="*60)
    print("📊 INVESTIGATION REPORT")
    print("="*60)
    print(f"Total Investigations: {total}")
    print(f"Supported Claims: {supported} ({supported/total*100:.1f} %)")
    print(f"Refuted Claims: {refuted} ({refuted/total*100:.1f} %)")
    print(f"Average Confidence: {avg_confidence*100:.1f}%")
    print("="*60)

    # Common layout for dark theme
    common_layout = dict(
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0.1)',
        font=dict(color='white'),
        title_font_size=18
    )

    # Helper to display static image
    def display_static_image(fig, title):
        img_bytes = pio.to_image(fig, format='png', width=900, height=500, scale=2)
        display(HTML(f"<h3>{title}</h3>"))
        display(Image.open(io.BytesIO(img_bytes)))

    # 1. Investigations Over Time (Confidence Timeline)
    fig_timeline = px.line(
        df, y='confidence', x=df.index, title='Investigations Over Time (Confidence)',
        markers=True, line_shape='linear', color_discrete_sequence=['#63b3ed']
    )
    fig_timeline.update_traces(
        marker=dict(size=10, color=df['verdict'].map({'SUPPORTED': '#48bb78', 'REFUTED': '#f56565'})),
        line=dict(width=3)
    )
    fig_timeline.update_layout(common_layout, title_text='📈 Investigations Over Time')
    display_static_image(fig_timeline, 'Investigations Over Time')

    # 2. Verdict Distribution (Pie Chart)
    verdict_counts = df['verdict'].value_counts().reset_index()
    verdict_counts.columns = ['Verdict', 'Count']
    fig_pie = px.pie(
        verdict_counts, values='Count', names='Verdict', title='Verdict Distribution',
        color_discrete_sequence=['#48bb78', '#f56565']
    )
    fig_pie.update_layout(common_layout, title_text='⚖️ Verdict Distribution')
    display_static_image(fig_pie, 'Verdict Distribution')

    # 3. Confidence Trend (Rolling Average)
    df['rolling_avg_confidence'] = df['confidence'].rolling(window=3, min_periods=1).mean()
    fig_trend = px.line(
        df, y='rolling_avg_confidence', x=df.index, title='Confidence Trend (Rolling Average)',
        line_shape='spline', color_discrete_sequence=['#FFEAA7']
    )
    fig_trend.update_layout(common_layout, title_text='📉 Confidence Trend')
    display_static_image(fig_trend, 'Confidence Trend')

    # 4. Investigation Details (Table)
    fig_table = go.Figure(data=[go.Table(
        header=dict(
            values=['Time', 'Claim', 'Verdict', 'Confidence'],
            fill_color='rgba(0,0,0,0.2)',
            font=dict(color='white', size=12),
            align='left'
        ),
        cells=dict(
            values=[
                df['timestamp'],
                df['claim'].str[:60] + "...", # Increased claim preview length
                df['verdict'],
                (df['confidence']*100).round(1).astype(str) + "%"
            ],
            fill_color='rgba(0,0,0,0.1)',
            font=dict(color='white', size=10),
            align='left',
            height=30
        )
    )])
    fig_table.update_layout(
        title_text="📋 Investigation Details",
        title_font_size=18,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0.1)',
        font=dict(color='white')
    )
    display_static_image(fig_table, 'Investigation Details')


# Modified show_history_button to use a robust HTML button and Colab callback
def show_history_button_static(dashboard):
    """Add history button to dashboard, using a robust HTML button and Colab callback."""

    # Define the function that will be called by JavaScript
    # This encapsulates the original on_click logic
    def _generate_report_callback():
        with dashboard.output_area:
            clear_output(wait=True)
            if not dashboard.investigation_history:
                print("No investigations yet!")
                return
            # Call the existing report generation function
            create_investigation_report_static(dashboard.investigation_history)

    # Register the Python function with Colab's output system
    # The string 'generate_static_report_callback' becomes the JavaScript function name
    # accessible via google.colab.output.send
    output.register_callback('generate_static_report_callback', _generate_report_callback)

    # Create a raw HTML button that triggers the registered callback via JavaScript
    # The style approximates ipywidgets' 'warning' button_style.
    button_html = """
    <button id="static_report_button"
            style="background-color: #ffc107; color: #212529; border: 1px solid #ffc107;
                   padding: 8px 12px; text-align: center; text-decoration: none;
                   display: inline-block; font-size: 14px; margin: 4px 2px;
                   cursor: pointer; border-radius: 0.25rem;
                   font-family: -apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica Neue,Arial,sans-serif;"
            onclick="google.colab.output.send('generate_static_report_callback')">
        📊 View Investigation History (Static)
    </button>
    """
    # Return an IPython.display.HTML object to render the button in the notebook output
    return HTML(button_html)

# Display the new static history button
display(show_history_button_static(dashboard))
